# AI Agents MCP & Tool Design

## Model Context Protocol (MCP)

MCP (Anthropic, 2024) is an **open standard** for connecting AI models to external data sources and tools.

**Problem it solves**: Every tool integration was custom. MCP standardizes how models connect to the world.

```
Without MCP: LLM ←→ custom code ←→ filesystem
             LLM ←→ custom code ←→ GitHub
             LLM ←→ custom code ←→ database

With MCP:    LLM (Claude/Cursor/etc)
             ↕ MCP Protocol
             MCP Servers (filesystem, GitHub, Postgres, Slack, ...)
```

---

## MCP Architecture

| Component | Role |
|-----------|------|
| **Host** | Application running the LLM (Claude Desktop, Cursor) |
| **Client** | Protocol client inside the host one per server |
| **Server** | Lightweight process exposing tools/resources |

Communication: JSON-RPC 2.0 over stdio or HTTP+SSE

---

## MCP Primitives

### Tools
Functions the model can call (like function calling):
```json
{"name": "read_file", "description": "Read file contents", "inputSchema": {...}}
```

### Resources
Data the model can read (like RAG documents):
```json
{"uri": "file:///path/to/doc.txt", "mimeType": "text/plain"}
```

### Prompts
Reusable prompt templates with arguments:
```json
{"name": "summarize", "arguments": [{"name": "text", "required": true}]}
```

---

## Popular MCP Servers

| Server | What it exposes |
|--------|----------------|
| `@modelcontextprotocol/server-filesystem` | Read/write local files |
| `@modelcontextprotocol/server-github` | Repos, issues, PRs |
| `@modelcontextprotocol/server-postgres` | Query PostgreSQL |
| `@modelcontextprotocol/server-brave-search` | Web search |
| `@modelcontextprotocol/server-slack` | Send/read Slack messages |
| `mcp-server-git` | Git operations |
| `mcp-pandoc` | Document conversion |

---

## Tool Design Best Practices

| Practice | Why |
|----------|-----|
| Clear descriptions | LLM uses description to decide when to call |
| Specific parameter names | `user_id` not `id` |
| Return structured data | JSON over free text |
| Include error info | Return error messages, not exceptions |
| Idempotent when possible | Safe to call multiple times |
| Atomic operations | One tool = one action |

In [1]:
# ── Build an MCP Server in Python ─────────────────────────────────────────────
# pip install mcp
# Save this as mcp_server.py and run: python mcp_server.py

mcp_server_code = '''
import asyncio
import json
from pathlib import Path
from mcp.server import Server
from mcp.server.stdio import stdio_server
from mcp import types

server = Server("file-tools")

@server.list_tools()
async def list_tools() -> list[types.Tool]:
    return [
        types.Tool(
            name="read_file",
            description="Read the contents of a text file",
            inputSchema={
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Absolute file path"}
                },
                "required": ["path"]
            }
        ),
        types.Tool(
            name="list_directory",
            description="List files in a directory",
            inputSchema={
                "type": "object",
                "properties": {
                    "path": {"type": "string", "description": "Directory path"}
                },
                "required": ["path"]
            }
        )
    ]

@server.call_tool()
async def call_tool(name: str, arguments: dict) -> list[types.TextContent]:
    if name == "read_file":
        path = Path(arguments["path"])
        if not path.exists():
            return [types.TextContent(type="text", text=f"Error: File not found: {path}")]
        content = path.read_text()
        return [types.TextContent(type="text", text=content)]
    
    elif name == "list_directory":
        path = Path(arguments["path"])
        if not path.is_dir():
            return [types.TextContent(type="text", text=f"Error: Not a directory: {path}")]
        files = [str(f) for f in path.iterdir()]
        return [types.TextContent(type="text", text=json.dumps(files, indent=2))]

async def main():
    async with stdio_server() as (read_stream, write_stream):
        await server.run(read_stream, write_stream, server.create_initialization_options())

if __name__ == "__main__":
    asyncio.run(main())
'''

with open("/tmp/mcp_server_example.py", "w") as f:
    f.write(mcp_server_code)

print("MCP server code written to /tmp/mcp_server_example.py")
print("\nTo use with Claude Desktop, add to ~/Library/Application Support/Claude/claude_desktop_config.json:")
print(json_config := """{
  "mcpServers": {
    "file-tools": {
      "command": "python",
      "args": ["/tmp/mcp_server_example.py"]
    }
  }
}""")
print(json_config)

MCP server code written to /tmp/mcp_server_example.py

To use with Claude Desktop, add to ~/Library/Application Support/Claude/claude_desktop_config.json:
{
  "mcpServers": {
    "file-tools": {
      "command": "python",
      "args": ["/tmp/mcp_server_example.py"]
    }
  }
}
{
  "mcpServers": {
    "file-tools": {
      "command": "python",
      "args": ["/tmp/mcp_server_example.py"]
    }
  }
}


In [2]:
# ── MCP Client (calling an MCP server programmatically) ──────────────────────
# pip install mcp anthropic

mcp_client_code = '''
import asyncio
import anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_with_mcp():
    server_params = StdioServerParameters(
        command="python",
        args=["/tmp/mcp_server_example.py"]
    )
    
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # List available tools
            tools_response = await session.list_tools()
            tools = tools_response.tools
            print(f"Available tools: {[t.name for t in tools]}")
            
            # Convert to Anthropic format
            anthropic_tools = [{
                "name": t.name,
                "description": t.description,
                "input_schema": t.inputSchema
            } for t in tools]
            
            # Call Claude with MCP tools
            client = anthropic.Anthropic()
            response = client.messages.create(
                model="claude-sonnet-4-6",
                max_tokens=1024,
                tools=anthropic_tools,
                messages=[{"role": "user", "content": "List files in /tmp"}]
            )
            
            # Handle tool calls
            for block in response.content:
                if block.type == "tool_use":
                    result = await session.call_tool(block.name, block.input)
                    print(f"Tool {block.name} result: {result.content[0].text[:200]}")

# asyncio.run(run_with_mcp())
print("MCP client pattern ready")
'''

print(mcp_client_code)


import asyncio
import anthropic
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

async def run_with_mcp():
    server_params = StdioServerParameters(
        command="python",
        args=["/tmp/mcp_server_example.py"]
    )

    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()

            # List available tools
            tools_response = await session.list_tools()
            tools = tools_response.tools
            print(f"Available tools: {[t.name for t in tools]}")

            # Convert to Anthropic format
            anthropic_tools = [{
                "name": t.name,
                "description": t.description,
                "input_schema": t.inputSchema
            } for t in tools]

            # Call Claude with MCP tools
            client = anthropic.Anthropic()
            response = client.messages.create(
       

In [3]:
# ── Tool Design Patterns ──────────────────────────────────────────────────────
from typing import Any

# Pattern 1: Always return structured JSON
def good_tool(query: str) -> dict:
    """Search for information. Returns structured results."""
    try:
        # ... actual logic ...
        return {
            "success": True,
            "results": [{"title": "...", "url": "...", "snippet": "..."}],
            "total_results": 10
        }
    except Exception as e:
        return {"success": False, "error": str(e)}

# Pattern 2: Validate inputs before expensive operations
def safe_file_read(path: str, max_size_mb: float = 10.0) -> dict:
    """Read a file safely with size limits."""
    import os
    if not os.path.exists(path):
        return {"error": f"File not found: {path}"}
    
    size_mb = os.path.getsize(path) / (1024 * 1024)
    if size_mb > max_size_mb:
        return {"error": f"File too large ({size_mb:.1f}MB). Max: {max_size_mb}MB"}
    
    try:
        with open(path) as f:
            content = f.read()
        return {"content": content, "size_bytes": len(content)}
    except PermissionError:
        return {"error": "Permission denied"}

# Pattern 3: Tool with dry-run capability
def send_email(to: str, subject: str, body: str, dry_run: bool = True) -> dict:
    """Send an email. Use dry_run=True to preview without sending."""
    if dry_run:
        return {
            "dry_run": True,
            "would_send": {"to": to, "subject": subject, "body_preview": body[:100]}
        }
    # Actual send logic here
    return {"sent": True, "message_id": "msg-123"}

# Test
print(send_email("user@example.com", "Test", "Hello world", dry_run=True))

{'dry_run': True, 'would_send': {'to': 'user@example.com', 'subject': 'Test', 'body_preview': 'Hello world'}}


## Additional Learning Resources

### MCP
- [MCP Official Docs](https://modelcontextprotocol.io/)
- [MCP Specification](https://spec.modelcontextprotocol.io/)
- [MCP GitHub](https://github.com/modelcontextprotocol)
- [MCP Servers Registry](https://github.com/modelcontextprotocol/servers)

### Tool Use
- [Anthropic Tool Use Guide](https://docs.anthropic.com/en/docs/build-with-claude/tool-use)
- [OpenAI Function Calling](https://platform.openai.com/docs/guides/function-calling)
- [Toolformer Paper (Schick et al., 2023)](https://arxiv.org/abs/2302.04761)

### Computer Use
- [Anthropic Computer Use](https://docs.anthropic.com/en/docs/build-with-claude/computer-use)
- [Browser Use (GitHub)](https://github.com/browser-use/browser-use)